# Every-row anchor masked TVT model

Her uygun kuyu satiri deterministik bir anchor olur. Anchor'a kadar TVT context olarak verilir; sonraki horizon maskelenir ve sequence-to-sequence GRU tarafindan tahmin edilir. Validation ve reserved test skoru gercek `TVT_input` maskesiyle hesaplanir.

In [1]:
import copy
import random
from typing import Literal

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

from training import (
    MaskedTVTSequenceDataset,
    RNN_DROP_COLUMNS,
    RNN_FEATURE_COLUMNS,
    prepare_rnn_features,
)
from utils import (
    get_all_test_ids,
    get_all_training_ids,
    test_folder,
    train_folder,
)

SEED = 42
VALID_SIZE = 0.15
CONTEXT_LENGTH = 64
FORECAST_LENGTH = 64
ANCHOR_STRIDE = 1
BATCH_SIZE = 128
NUM_EPOCHS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


In [2]:
all_training_ids = get_all_training_ids()
reserved_test_ids = set(get_all_test_ids())
modeling_ids = sorted(set(all_training_ids) - reserved_test_ids)

train_well_ids, valid_well_ids = train_test_split(
    modeling_ids,
    test_size=VALID_SIZE,
    random_state=SEED,
    shuffle=True,
)

assert set(train_well_ids).isdisjoint(valid_well_ids)
assert set(train_well_ids).isdisjoint(reserved_test_ids)
print(f"Train wells: {len(train_well_ids)}")
print(f"Validation wells: {len(valid_well_ids)}")
print(f"Reserved test wells: {len(reserved_test_ids)}")

Train wells: 654
Validation wells: 116
Reserved test wells: 3


In [3]:
train_wells = prepare_rnn_features(
    training_ids=train_well_ids,
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
)
valid_wells = prepare_rnn_features(
    training_ids=valid_well_ids,
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
)
test_wells = prepare_rnn_features(
    training_ids=sorted(reserved_test_ids),
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
    data_dir=test_folder,
    target_data_dir=train_folder,
)

len(train_wells), len(valid_wells), len(test_wells)

(654, 116, 3)

In [4]:
MASK_FEATURE_COLUMNS = [
    "GR_isna",
    "GR_typewell_imputed_mask",
    "GR_neighbor_imputed_mask",
]
CONTINUOUS_FEATURE_COLUMNS = [
    column for column in RNN_FEATURE_COLUMNS
    if column not in MASK_FEATURE_COLUMNS
]

feature_scaler = StandardScaler()
target_scaler = StandardScaler()
for well in train_wells.values():
    feature_scaler.partial_fit(well.data[CONTINUOUS_FEATURE_COLUMNS])
    target_scaler.partial_fit(well.data[["TVT"]])

target_mean = float(target_scaler.mean_[0])
target_scale = float(target_scaler.scale_[0])

def apply_scalers(prepared_wells):
    for well in prepared_wells.values():
        well.data.loc[:, CONTINUOUS_FEATURE_COLUMNS] = (
            feature_scaler.transform(
                well.data[CONTINUOUS_FEATURE_COLUMNS]
            )
        )
        well.data.loc[:, "TVT_scaled"] = target_scaler.transform(
            well.data[["TVT"]]
        ).ravel()
        well.data.loc[:, "observed_TVT_scaled"] = 0.0
        observed_mask = well.data["observed_TVT"].notna()
        well.data.loc[observed_mask, "observed_TVT_scaled"] = (
            well.data.loc[observed_mask, "observed_TVT"] - target_mean
        ) / target_scale

apply_scalers(train_wells)
apply_scalers(valid_wells)
apply_scalers(test_wells)
target_mean, target_scale

(11502.671923024185, 637.698604177825)

In [9]:
train_wells

{'b38e3116': <training_data.TrainingData at 0x11ee881a0>,
 '39b51819': <training_data.TrainingData at 0x11edce990>,
 '95b559e7': <training_data.TrainingData at 0x11edced50>,
 '044af7d1': <training_data.TrainingData at 0x10b1069e0>,
 '445af6c2': <training_data.TrainingData at 0x11edf62c0>,
 '4cecf7b3': <training_data.TrainingData at 0x11ee22b10>,
 '0a57a29c': <training_data.TrainingData at 0x11ee59bf0>,
 '74b6186b': <training_data.TrainingData at 0x11ee58e20>,
 '25050f63': <training_data.TrainingData at 0x11ea6d950>,
 'febb4411': <training_data.TrainingData at 0x11ea6e650>,
 '6767c111': <training_data.TrainingData at 0x11ee85a90>,
 '57f05c51': <training_data.TrainingData at 0x11ee85d60>,
 '43c7f85a': <training_data.TrainingData at 0x11f3b03d0>,
 '55efee7f': <training_data.TrainingData at 0x11f3b0130>,
 '01982c1d': <training_data.TrainingData at 0x10b3d7860>,
 'd63196fe': <training_data.TrainingData at 0x11ee7c4d0>,
 '87695469': <training_data.TrainingData at 0x11ee7c350>,
 '4f3eb9e9': <

In [5]:
def make_synthetic_datasets(prepared_wells):
    return {
        well_id: MaskedTVTSequenceDataset(
            data=well.data,
            feature_columns=RNN_FEATURE_COLUMNS,
            target_column="TVT_scaled",
            context_length=CONTEXT_LENGTH,
            forecast_length=FORECAST_LENGTH,
            anchor_stride=ANCHOR_STRIDE,
            mode="synthetic",
        )
        for well_id, well in prepared_wells.items()
        if len(well.data) >= CONTEXT_LENGTH + FORECAST_LENGTH
    }

def make_real_ps_datasets(prepared_wells):
    return {
        well_id: MaskedTVTSequenceDataset(
            data=well.data,
            feature_columns=RNN_FEATURE_COLUMNS,
            target_column="TVT_scaled",
            mode="real",
            prediction_mask_column="is_prediction_row",
            observed_context_column="observed_TVT_scaled",
        )
        for well_id, well in prepared_wells.items()
    }

train_datasets = make_synthetic_datasets(train_wells)
valid_datasets = make_real_ps_datasets(valid_wells)
test_datasets = make_real_ps_datasets(test_wells)

example_well_id = next(iter(train_datasets))
sample_0 = train_datasets[example_well_id][0]
sample_1 = train_datasets[example_well_id][1]
print(f"Example well: {example_well_id}")
print(f"Number of anchors: {len(train_datasets[example_well_id])}")
print(f"First anchors: {sample_0['anchor']}, {sample_1['anchor']}")
print(f"Input shape: {tuple(sample_0['features'].shape)}")
print(f"Known/loss rows: {int(sample_0['known_mask'].sum())}/{int(sample_0['loss_mask'].sum())}")

Example well: b38e3116
Number of anchors: 7825
First anchors: 63, 64
Input shape: (128, 22)
Known/loss rows: 64/64


In [6]:
RNN_TYPES = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}

class MaskedTVTModel(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int = 64,
        rnn_type: Literal["rnn", "gru", "lstm"] = "gru",
        num_layers: int = 1,
        dropout: float = 0.2,
    ):
        super().__init__()
        if rnn_type not in RNN_TYPES:
            raise ValueError(f"Invalid RNN type: {rnn_type}")
        self.rnn = RNN_TYPES[rnn_type](
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        sequence_output, _ = self.rnn(x)
        return self.head(sequence_output).squeeze(-1)

model = MaskedTVTModel(
    input_size=len(RNN_FEATURE_COLUMNS) + 2,
    hidden_size=64,
    rnn_type="gru",
).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-3, weight_decay=1e-5
)

with torch.no_grad():
    smoke_input = sample_0["features"].unsqueeze(0).to(device)
    smoke_output = model(smoke_input)
assert smoke_output.shape == (1, CONTEXT_LENGTH + FORECAST_LENGTH)
smoke_output.shape

torch.Size([1, 128])

In [7]:
def masked_mse_loss(predictions, targets, loss_mask):
    return ((predictions - targets) ** 2)[loss_mask].mean()

def train_one_epoch(model, datasets, optimizer):
    model.train()
    squared_error_sum = 0.0
    observation_count = 0
    well_ids = list(datasets)
    random.shuffle(well_ids)

    for well_id in well_ids:
        loader = DataLoader(
            datasets[well_id], batch_size=BATCH_SIZE,
            shuffle=True, num_workers=0,
        )
        for batch in loader:
            features = batch["features"].to(device)
            targets = batch["target"].to(device)
            loss_mask = batch["loss_mask"].to(device)
            optimizer.zero_grad(set_to_none=True)
            predictions = model(features)
            loss = masked_mse_loss(predictions, targets, loss_mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            errors_tvt = (predictions - targets) * target_scale
            squared_error_sum += torch.sum(errors_tvt[loss_mask] ** 2).item()
            observation_count += int(loss_mask.sum().item())

    return float(np.sqrt(squared_error_sum / observation_count))

@torch.no_grad()
def evaluate_real_ps(model, datasets):
    model.eval()
    squared_error_sum = 0.0
    observation_count = 0
    for dataset in datasets.values():
        sample = dataset[0]
        features = sample["features"].unsqueeze(0).to(device)
        targets = sample["target"].unsqueeze(0).to(device)
        loss_mask = sample["loss_mask"].unsqueeze(0).to(device)
        predictions = model(features)
        errors_tvt = (predictions - targets) * target_scale
        squared_error_sum += torch.sum(errors_tvt[loss_mask] ** 2).item()
        observation_count += int(loss_mask.sum().item())
    if observation_count == 0:
        raise ValueError("Gercek PS sonrasinda skorlanacak satir yok.")
    return float(np.sqrt(squared_error_sum / observation_count))

In [8]:
history = []
best_valid_rmse = float("inf")
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    train_rmse = train_one_epoch(model, train_datasets, optimizer)
    valid_rmse = evaluate_real_ps(model, valid_datasets)
    history.append({
        "epoch": epoch,
        "synthetic_train_rmse": train_rmse,
        "real_ps_valid_rmse": valid_rmse,
    })
    if valid_rmse < best_valid_rmse:
        best_valid_rmse = valid_rmse
        best_state = copy.deepcopy(model.state_dict())
    print(
        f"Epoch {epoch:02d} | synthetic train RMSE: {train_rmse:.4f} | "
        f"real-PS valid RMSE: {valid_rmse:.4f}"
    )

if best_state is not None:
    model.load_state_dict(best_state)
test_rmse = evaluate_real_ps(model, test_datasets)
print(f"Masked reserved test RMSE: {test_rmse:.4f}")
pd.DataFrame(history)

Epoch 01 | synthetic train RMSE: 104.5585 | real-PS valid RMSE: 230.5854
Epoch 02 | synthetic train RMSE: 57.9433 | real-PS valid RMSE: 144.7554
Epoch 03 | synthetic train RMSE: 51.5176 | real-PS valid RMSE: 102.1510
Epoch 04 | synthetic train RMSE: 46.9470 | real-PS valid RMSE: 121.8868
Epoch 05 | synthetic train RMSE: 45.2735 | real-PS valid RMSE: 159.2047
Masked reserved test RMSE: 121.5597


,epoch,synthetic_train_rmse,real_ps_valid_rmse
0,1,104.558487,230.585383
1,2,57.943283,144.755357
2,3,51.517596,102.151048
3,4,46.946955,121.886831
4,5,45.273457,159.204664
